<a href="https://colab.research.google.com/github/Murcha1990/HSE_FitPredict_AI_School/blob/main/ML/18_11/02_practice_ames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика: деревья, лес и бустинг на Ames House Prices

**Задача:** предсказать цену продажи дома (`SalePrice`) в городе Эймс (штат Айова) по 79 признакам: площадь, качество отделки, район, год постройки, гараж, подвал и т. д.

**Метрика:** RMSE в логарифмах цены (функция `score` ниже) — **чем меньше, тем лучше**.

**Правила**

- Разбиение на train и test зафиксировано — не меняйте `random_state`.
- Подбирайте всё (предобработку, признаки, гиперпараметры) только по train, например кросс-валидацией.
- Test — только для итоговой проверки. Свой лучший результат на test напишите в чат.

**Две части**

1. **20 минут** — только sklearn: дерево, случайный лес, градиентный бустинг.
2. **30 минут** — CatBoost.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (9, 4)

## Данные

In [ ]:
ames = fetch_openml(name='house_prices', as_frame=True, parser='auto')
df = ames.frame.drop(columns='Id')
print(df.shape)
df.head()

(1460, 80)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
X = df.drop(columns='SalePrice')
y = df['SalePrice']

# Разбиение фиксировано: у всех одинаковый test, поэтому результаты можно сравнивать
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
print(f'train: {X_train.shape}, test: {X_test.shape}')
print(f'числовых признаков: {len(num_cols)}, категориальных: {len(cat_cols)}')

train: (1095, 79), test: (365, 79)
числовых признаков: 36, категориальных: 43


In [ ]:
def score(y_true, y_pred):
    """RMSE в логарифмах цены — метрика соревнования Kaggle House Prices.
    Ошибка 0.10 примерно означает «ошибаемся в среднем на 10%»."""
    y_pred = np.maximum(y_pred, 1)
    return root_mean_squared_error(np.log1p(y_true), np.log1p(y_pred))

## Базовое решение

Случайный лес только на числовых признаках, пропуски заполнены медианой. Ваша задача — улучшить этот результат.

In [ ]:
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline.fit(X_train[num_cols], y_train)
print('baseline, test:', round(score(y_test, baseline.predict(X_test[num_cols])), 4))

baseline, test: 0.1474


# Часть 1 (20 минут): sklearn

**1. Исследуйте данные.** Есть ли проблемы с качеством данных? Посмотрите на:

- пропуски: где их много и что они означают;
- распределение целевой переменной;
- выбросы;
- типы признаков: все ли числовые признаки на самом деле числовые, все ли категориальные — неупорядоченные?

**2. Сравните модели.** Обучите дерево, случайный лес и градиентный бустинг с параметрами по умолчанию на всех признаках. Какая модель сильнее переобучается?

**3. Улучшите качество.** Используйте то, что нашли в пункте 1: предобработку, новые признаки, подбор гиперпараметров.

Полезные инструменты: `ColumnTransformer`, `SimpleImputer`, `OrdinalEncoder` / `OneHotEncoder`, `TransformedTargetRegressor`, `GridSearchCV`.

In [ ]:
# 1. Исследование данных

In [ ]:
# 2. Дерево, лес и бустинг с параметрами по умолчанию

In [ ]:
# 3. Ваше лучшее решение на sklearn

# Часть 2 (30 минут): CatBoost

1. Обучите `CatBoostRegressor` на исходных признаках: передайте категориальные признаки через `cat_features`.
   В категориальных признаках не должно быть пропусков — заполните их строкой, например `'Missing'`.
2. Используйте раннюю остановку: отрежьте от train валидационную выборку и передайте её в `eval_set`.
3. Сравните CatBoost с вашим лучшим решением на sklearn.
4. Добавьте признаки, которые вы придумали в части 1. Помогают ли они CatBoost?
5. Посмотрите на важность признаков (`get_feature_importance`). Совпадает ли она с вашими ожиданиями?

In [ ]:
!pip install catboost
from catboost import CatBoostRegressor

In [ ]:
# 1–2. CatBoost на исходных признаках с ранней остановкой

In [ ]:
# 3–5. Сравнение, новые признаки, важность признаков